# Build the ViT-Motion Kaggle Dataset (run once)

Pulls `nycu-apl/vit-motion-dataset` from Hugging Face into `/kaggle/working`,
keeping only what the model needs (`samples.csv` + `rgb/`), then you save it
as a **persistent Kaggle Dataset** so training/eval never re-download it.

**Before running:** in the notebook sidebar set **Internet = ON**.
(Settings -> Internet). A GPU is not needed for this notebook.


In [ ]:
# 1) Download only the model-facing files (skips depth/ and summary.json to save space)
!pip -q install "huggingface_hub>=0.23" hf_transfer >/dev/null
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import snapshot_download

local = snapshot_download(
    repo_id="nycu-apl/vit-motion-dataset",
    repo_type="dataset",
    local_dir="/kaggle/working/vit-motion-dataset",
    allow_patterns=["*/samples.csv", "*/rgb/*"],   # drop this line to also fetch depth/
    max_workers=8,
)
print("Downloaded to:", local)

In [ ]:
# 2) Sanity check: how many experiments and how big
import glob
csvs = glob.glob("/kaggle/working/vit-motion-dataset/**/samples.csv", recursive=True)
print("experiments (samples.csv found):", len(csvs))
print("example:", csvs[0] if csvs else "NONE")
!du -sh /kaggle/working/vit-motion-dataset

## 3) Persist it as a Kaggle Dataset

Two ways — pick one:

**A. From notebook output (simplest).** Click **Save Version** (Quick Save) at
top-right. When it finishes, open this notebook's **Output** tab, and use
**New Dataset -> from notebook output**. Name it e.g. `vit-motion-dataset`.

**B. Kaggle API.** Add your `kaggle.json` token via **Add-ons -> Secrets** (or
the Kaggle CLI locally) and run the optional cell below.

Either way, in the run notebook the dataset is auto-detected under
`/kaggle/input/...` — the exact slug does not matter.


In [ ]:
# OPTIONAL (way B) — push as a Kaggle Dataset via the API.
# Requires ~/.kaggle/kaggle.json (Kaggle account -> Create New API Token).
# import json, os, pathlib, subprocess
# root = "/kaggle/working/vit-motion-dataset"
# meta = {"title": "vit-motion-dataset", "id": "YOUR_USERNAME/vit-motion-dataset",
#         "licenses": [{"name": "CC-BY-4.0"}]}
# pathlib.Path(root, "dataset-metadata.json").write_text(json.dumps(meta))
# subprocess.run(["kaggle", "datasets", "create", "-p", root, "--dir-mode", "zip"], check=True)